# Costmap 訂單排程與 Sower 教學
本 notebook 使用新版 scheduler 引擎。先用 A* 建立距離／時間矩陣，再以 sigmoid 產生訂單權重，最後觀察 PMX 基因演算法與跨世界菁英共享。
SAW 在此指揭露書的 sower，不是 Simple Additive Weighting。範例是靜態單樓層、單機器人，不直接控制硬體。

In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'app.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from PIL import Image
import matplotlib.pyplot as plt
from scheduler.core import Costmap, Order, prepare, weights, Objective, evolve


## 1 地圖與節點
黑色牆壁先按機器人半徑膨脹。8 鄰接 A* 不能斜穿牆角；找不到通道時會明確失敗。

In [ ]:
meta = json.loads((ROOT / 'data/map/hotel.json').read_text())
grid = Costmap(Image.open(ROOT / 'data/map/hotel.png'), resolution=0.1, radius=0.2)
orders = [Order(f'Room {101+i}', x, y, importance=5+i%5, waited=20*i, patience=300, risk=0.5) for i,(x,y) in enumerate(meta['points'])]
matrix = prepare(grid, orders, tuple(meta['start']))
plt.figure(figsize=(10,6))
plt.imshow(grid.pixels, cmap='gray', vmin=0, vmax=255)
plt.scatter([o.x for o in orders], [o.y for o in orders], c='orange')
plt.title('Hotel costmap and delivery nodes')
plt.show()


## 2 訂單權重
$w_j=1+9\sigma(z_j)$，$\sigma(z)=1/(1+e^{-z})$。
重要度、等待／耐久比、棄單風險、起點 A* 距離和附近慢速因素組合成 z。完整公式見 docs/algorithm.md。


In [ ]:
rows = weights(orders, matrix, cruise=0.8)
for row in rows: print(row['id'], round(row['weight'],3))
print('A* distance matrix (m):')
print(matrix['distance'].round(1))


## 3 營運 loss 與 Sower
loss 由完成時間、加權平均抵達時間、逾時風險與總距離構成。SAW 開啟後各世界使用獨立 RNG，每 10 代交換菁英。
PMX 保證排列不重複，保留前 10% 菁英，swap / inversion 後重新評分。禁止順位用匹配修復，不可行則報錯。

In [ ]:
objective = Objective(orders, matrix, rows, return_to_start=True)
result = evolve(objective, generations=60, population=50, worlds=4, sower=True, interval=10, seed=42)
print('Best loss:', result['best']['loss'])
print('Schedule:', [s['id'] for s in result['best']['schedule']])
print('Sower events:', result['events'])


## 4 進化曲線與避牆路徑
每個世界最佳 loss 因菁英保留而不增加。Sower 不保證在每個問題都優於單世界；比較時應注意總評估預算。

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(14,5))
for w in range(4):
    axes[0].plot([r['generation'] for r in result['history']], [r['worlds'][w]['loss'] for r in result['history']], label=f'World {w+1}')
for event in result['events']: axes[0].axvline(event['generation'], color='gray', alpha=0.2, linestyle='--')
axes[0].legend();axes[0].set(xlabel='Generation',ylabel='Loss')
axes[1].imshow(grid.pixels,cmap='gray',vmin=0,vmax=255)
route=result['best']['route']
for i,j in zip(route,route[1:]):
    path=matrix['paths'][f'{i}:{j}']
    axes[1].plot([p[0] for p in path],[p[1] for p in path],color='seagreen')
for s in result['best']['schedule']:
    p=matrix['points'][s['node']];axes[1].annotate(str(s['position']),p,color='darkorange')
axes[1].set_title('A* collision-free grid route')
plt.show()
